# 01 - Exploration des Donnees EpiGraph

Ce notebook permet d'explorer le dataset epidemiologique stocke dans Neo4j.

## Objectifs
- Se connecter a Neo4j depuis Python
- Explorer la structure du graphe
- Comprendre les types de noeuds et relations

In [1]:
# Installation des dependances si necessaire
!pip install neo4j pandas matplotlib

In [11]:
import os
from neo4j import GraphDatabase
import pandas as pd

# Configuration connexion Neo4j
NEO4J_URI = os.environ.get('NEO4J_URI', 'bolt://neo4j:7687')
NEO4J_USER = os.environ.get('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print('Connexion Neo4j etablie')

Connexion Neo4j etablie


In [4]:
def run_query(query, params=None):
    """Execute une requete Cypher et retourne un DataFrame."""
    with driver.session() as session:
        result = session.run(query, params or {})
        return pd.DataFrame([dict(record) for record in result])

## 1. Statistiques globales du graphe

In [12]:
# Compter les noeuds par type
query = """
MATCH (n)
RETURN labels(n)[0] AS Type, count(n) AS Count
ORDER BY Count DESC
"""
df_nodes = run_query(query)
print('Types de noeuds dans le graphe:')
df_nodes

Types de noeuds dans le graphe:


,Type,Count
0,Patient,50045
1,Hospital,111
2,Event,100
3,Location,30
4,Disease,5
5,Vaccine,4
6,Database,1
7,Message,1


In [13]:
# Compter les relations par type
query = """
MATCH ()-[r]->()
RETURN type(r) AS Type, count(r) AS Count
ORDER BY Count DESC
"""
df_rels = run_query(query)
print('Types de relations:')
df_rels

Types de relations:


,Type,Count
0,LIVES_IN,46479
1,HAS_DISEASE,40045
2,INFECTED_BY,22799
3,ATTENDED,21227
4,VACCINATED_WITH,14840
5,TREATED_AT,4561
6,TRANSFERS_TO,333
7,LOCATED_IN,111
8,HELD_IN,100
9,CONNECTED_TO,24


## 2. Distribution des patients

In [14]:
# Distribution par age
query = """
MATCH (p:Patient)
RETURN 
  CASE 
    WHEN p.age < 18 THEN '0-17'
    WHEN p.age < 40 THEN '18-39'
    WHEN p.age < 60 THEN '40-59'
    WHEN p.age < 75 THEN '60-74'
    ELSE '75+'
  END AS tranche_age,
  count(p) AS nombre
ORDER BY tranche_age
"""
df_age = run_query(query)
df_age

,tranche_age,nombre
0,0-17,8759
1,18-39,13955
2,40-59,13335
3,60-74,8257
4,75+,5739


In [15]:
# Distribution par statut vaccinal
query = """
MATCH (p:Patient)
RETURN p.vaccination_status AS statut, count(p) AS nombre
ORDER BY nombre DESC
"""
df_vax = run_query(query)
df_vax

,statut,nombre
0,unvaccinated,34945
1,partially,6966
2,fully,6528
3,boosted,1606


## 3. Repartition geographique

In [9]:
query = """
MATCH (p:Patient)-[:LIVES_IN]->(l:Location)
RETURN l.country AS pays, count(p) AS patients
ORDER BY patients DESC
"""
df_geo = run_query(query)
df_geo

,pays,patients
0,France,20109
1,United Kingdom,10745
2,Germany,5212
3,Italy,5211
4,Spain,5202


In [10]:
# Fermeture connexion
driver.close()
print('Exploration terminee')

Exploration terminee
